# Detection benchmarking

Precision / recall / F1 of each tool's taxon detection at species + genus rank,
for one selected dataset, against a unified threshold.

**Inputs** (all produced by `scripts/analysis/analysis_prep.py`):
- `results/metadata/analysis-prep/preprocessed/detection/<Tool>_<db>/<file>.csv` — standardized per-(sample, tool, db) tables
- `results/metadata/analysis-prep/preprocessed/detection/totals.csv` — total reads (or k-mers for Sourmash) used to normalize Kraken/Centrifuge/Centrifuger/Sourmash values
- `results/metadata/analysis-prep/ground_truth/*.csv` — unified species + genus truth tables


## 1. Imports

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Anchor cwd to bakeoff/ regardless of where the notebook is executed from.
BAKEOFF_ROOT = next(
    (d for d in [Path.cwd(), *Path.cwd().parents]
     if (d / 'config' / 'bakeoff_env.yaml').is_file()),
    None,
)
if BAKEOFF_ROOT is None:
    raise RuntimeError(f'Could not locate bakeoff/ root from {Path.cwd()}')
os.chdir(BAKEOFF_ROOT)
sys.path.insert(0, str(BAKEOFF_ROOT / 'scripts' / 'analysis'))

from utils.util import (
    apply_taxid_synonyms, presence_from_df, prf_counts, prf_scores,
)
from utils.results import find_latest_analysis_prep_ts, save_csv, save_fig
print(f'bakeoff root: {BAKEOFF_ROOT}')


## 2. Configuration

In [ ]:
SAVE = False

RESULTS_ROOT  = BAKEOFF_ROOT / 'results'
TS            = find_latest_analysis_prep_ts(RESULTS_ROOT)  # or pin: TS = "YYYYMMDD_HHMMSS"
PREP_DIR      = RESULTS_ROOT / 'metadata' / TS / 'analysis-prep'
DETECTION_DIR = PREP_DIR / 'preprocessed' / 'detection'
GT_DIR        = PREP_DIR / 'ground_truth'

THRESHOLD_PERCENT = 0.001   # taxon 'present' if value(%) >= threshold
RANKS             = ['species', 'genus']
SYNONYM_TAXID_MAP = {1578: 2742598}    # collapse synonymous taxa

# --- Dataset selector ---
DATASET = 'pacbio_low_input'
DATASETS = {
    # Each entry: project, technology, sample (raw, as they appear in the report tree)
    # and an optional `title` override for figure labels.
    'ont_zymo_kit':          dict(project='ZymoMockD6331', technology='ont',    sample='D6331_Zymo',    title='ZymoMockD6331 ONT Zymo prep'),
    'ont_qiagen_kit':        dict(project='ZymoMockD6331', technology='ont',    sample='D6331_Qiagen',  title='ZymoMockD6331 ONT Qiagen prep'),
    'pacbio_low_input':      dict(project='ZymoMockD6331', technology='pacbio', sample='SRR13128013',   title='ZymoMockD6331 PacBio low input'),
    'pacbio_standard_input': dict(project='ZymoMockD6331', technology='pacbio', sample='SRR13128014',   title='ZymoMockD6331 PacBio standard input'),
    'sim_ont':               dict(project='simulated',     technology='ont',    sample='sim_ont',       title='Simulated ONT'),
    'sim_pacbio':            dict(project='simulated',     technology='pacbio', sample='sim_pacbio',    title='Simulated PacBio'),
}
if DATASET not in DATASETS:
    raise ValueError(f'Unknown DATASET={DATASET!r}; choose from {sorted(DATASETS)}')
PROJECT, TECHNOLOGY, SAMPLE = (DATASETS[DATASET][k] for k in ('project', 'technology', 'sample'))
SAMPLE_LABEL  = f'{PROJECT}_{TECHNOLOGY}_{SAMPLE}'         # used in paths + filenames
DISPLAY_TITLE = DATASETS[DATASET].get('title', SAMPLE_LABEL)  # used in figure titles
PREP_FNAME    = f'{SAMPLE_LABEL}.csv'

TOOLS = [
    'Centrifuge_unified',  'Centrifuge_default',
    'Centrifuger_unified', 'Centrifuger_default',
    'Kraken2_unified',     'Kraken2_default',
    'Ganon2_unified',      'Ganon2_default',
    'Sourmash_unified',    'Sourmash_default',
    'Sylph_unified',       'Sylph_default',
]

THRESHOLD_DIR = RESULTS_ROOT / TS / 'detection' / f'threshold_{THRESHOLD_PERCENT}' / SAMPLE_LABEL
TABLES_DIR    = THRESHOLD_DIR / 'tables'
FIGURES_DIR   = THRESHOLD_DIR / 'figures'

print(f'{DATASET} -> {SAMPLE_LABEL}   threshold={THRESHOLD_PERCENT}%')
print(f'ts:    {TS}')
print(f'PREP:  {PREP_DIR}')
print(f'OUT:   {THRESHOLD_DIR}  (SAVE={SAVE})')


## 3. Ground truth

In [ ]:
def _gt_path(project, technology):
    if project == 'ZymoMockD6331':
        return GT_DIR / 'zymoD6331_gt.csv'
    if project == 'simulated':
        return GT_DIR / f'simulated_{technology}_gt.csv'
    raise ValueError(f'No ground truth configured for project={project!r}')

GT_FILE = _gt_path(PROJECT, TECHNOLOGY)
if not GT_FILE.is_file():
    raise FileNotFoundError(f'Ground truth missing: {GT_FILE} (run analysis_prep.py first)')

unified_gt = pd.read_csv(GT_FILE)
species_gt = unified_gt.query('rank == "species"').reset_index(drop=True)
genus_gt   = unified_gt.query('rank == "genus"').reset_index(drop=True)
print(f'truth: {GT_FILE.name}  species={len(species_gt)}  genus={len(genus_gt)}')


## 4. Load preprocessed per-tool tables

In [ ]:
# Load preprocessed CSVs (one per tool_db); skip ones missing on disk.
tool_tables_all = {
    t: pd.read_csv(DETECTION_DIR / t / PREP_FNAME)
    for t in TOOLS
    if (DETECTION_DIR / t / PREP_FNAME).is_file()
}
missing = [t for t in TOOLS if t not in tool_tables_all]
TOOLS = list(tool_tables_all)
print(f'loaded {len(TOOLS)} tool tables; missing: {missing or "none"}')

# Totals (kraken-style + sourmash); empty dict if no totals.csv on disk.
TOTALS_PATH = DETECTION_DIR / 'totals.csv'
if TOTALS_PATH.is_file():
    t_df = pd.read_csv(TOTALS_PATH).query(
        'project == @PROJECT and technology == @TECHNOLOGY and sample == @SAMPLE'
    )
    TOTAL_ASSIGNED_READS = dict(zip(t_df['tool_db'], t_df['total_for_normalization']))
else:
    TOTAL_ASSIGNED_READS = {}
print(f'totals: {len(TOTAL_ASSIGNED_READS)} entries')


## 5. Compute detection metrics

In [ ]:
def _clean_taxid(df):
    df = df.copy()
    df['taxid'] = pd.to_numeric(df['taxid'], errors='coerce').astype('Int64')
    return df.dropna(subset=['taxid']).reset_index(drop=True)

all_results = {}
for rank in RANKS:
    print(f'\n=== rank: {rank} ===')

    # Truth presence
    gt_src = species_gt if rank == 'species' else genus_gt
    gt = _clean_taxid(gt_src[['taxid', 'name']]).assign(value=1.0)
    gt = apply_taxid_synonyms(gt, abundance_col='value',
                              synonyms=SYNONYM_TAXID_MAP, collapse=True,
                              source_label=f'GT_{rank}')
    truth_present = presence_from_df(gt, threshold=0.0)
    print(f'  GT taxa: {len(gt)}')

    # Predictions + metrics
    tool_tables, rows = {}, []
    for tool_db in TOOLS:
        df = tool_tables_all[tool_db].query('rank == @rank').copy()
        if df.empty:
            tool_tables[tool_db] = df
            rows.append({'rank': rank, 'tool': tool_db,
                         'TP': 0, 'FP': 0, 'FN': len(truth_present), 'TN': 0,
                         'precision': 0.0, 'recall': 0.0, 'F1': 0.0})
            print(f'  {tool_db}: no rows')
            continue
        df = _clean_taxid(df)
        df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0.0)
        df = apply_taxid_synonyms(df, abundance_col='value',
                                  synonyms=SYNONYM_TAXID_MAP, collapse=True,
                                  source_label=f'{tool_db}_{rank}')
        tool_tables[tool_db] = df

        pred_present = presence_from_df(df, threshold=THRESHOLD_PERCENT)
        TP, FP, FN, TN = prf_counts(pred_present, truth_present)
        prec, rec, f1  = prf_scores(TP, FP, FN)
        rows.append({'rank': rank, 'tool': tool_db,
                     'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
                     'precision': prec, 'recall': rec, 'F1': f1})
        print(f'  {tool_db}: P={prec:.3f} R={rec:.3f} F1={f1:.3f}')

    metrics_df = pd.DataFrame(rows).sort_values('tool').reset_index(drop=True)
    out_csv = TABLES_DIR / f'metrics_{rank}.csv'
    save_csv(metrics_df, out_csv, SAVE, index=False)
    print(f'  -> {out_csv} (saved={SAVE})')

    all_results[rank] = {'metrics': metrics_df, 'tool_tables': tool_tables, 'ground_truth': gt}
    display(metrics_df)


## 6. Visualization

In [ ]:
# ---- Bar plots (matplotlib only) ----

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import numpy as np

BASE_TOOL_ORDER = []
for td in TOOLS:
    base = td.rsplit("_", 1)[0]
    if base not in BASE_TOOL_ORDER:
        BASE_TOOL_ORDER.append(base)

cmap = plt.get_cmap("tab10")
TOOL_COLORS = {tool: cmap(i % 10) for i, tool in enumerate(BASE_TOOL_ORDER)}

for rank in RANKS:
    print(f"\nGenerating plots for {rank}...")
    metrics_df = all_results[rank]["metrics"].copy()

    # Split "tool" into base tool and database type (default / unified)
    metrics_df[["base_tool", "db_type"]] = metrics_df["tool"].str.rsplit("_", n=1, expand=True)

    # Keep base tools in the predefined order, but only those present in this rank
    base_tools = [t for t in BASE_TOOL_ORDER if (metrics_df["base_tool"] == t).any()]

    # --- Three-panel plot: Precision, Recall, F1 ---
    fig, axes = plt.subplots(
        1, 3,
        figsize=(9, 4),
        sharey=True,
        gridspec_kw={"wspace": 0.02}
    )

    plt.subplots_adjust(
        left=0.07, right=0.98,
        top=0.82, bottom=0.25,
        wspace=0.02
    )

    metric_names = ["precision", "recall", "F1"]
    metric_titles = ["Precision", "Recall", "F1 Score"]

    width = 0.13
    x = np.arange(len(base_tools)) * 0.35
    offset = width * 0.6

    for i, (ax, metric, title) in enumerate(zip(axes, metric_names, metric_titles)):
        ax.set_facecolor("#f2f2f2" if i % 2 == 0 else "white")

        for j, base in enumerate(base_tools):
            color = TOOL_COLORS[base]

            row_def = metrics_df[
                (metrics_df["base_tool"] == base) &
                (metrics_df["db_type"] == "default")
            ]
            if not row_def.empty:
                ax.bar(
                    x[j] - offset,
                    row_def[metric].iloc[0],
                    width=width,
                    color=color,
                    alpha=0.35
                )

            row_uni = metrics_df[
                (metrics_df["base_tool"] == base) &
                (metrics_df["db_type"] == "unified")
            ]
            if not row_uni.empty:
                ax.bar(
                    x[j] + offset,
                    row_uni[metric].iloc[0],
                    width=width,
                    color=color,
                    alpha=0.95
                )

        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.set_xticks(x)
        ax.set_xticklabels(base_tools, rotation=65, ha="right", fontsize=12)
        # Shift tool names slightly to the right of each tick.
        _dx = mtransforms.ScaledTranslation(14 / 72, 0, fig.dpi_scale_trans)
        for _lbl in ax.get_xticklabels():
            _lbl.set_transform(_lbl.get_transform() + _dx)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.grid(axis="y", alpha=0.25)

    axes[0].set_ylabel("Score", fontsize=12)

    legend_handles = [
        plt.Line2D([0], [0], color="black", alpha=0.35, linewidth=8,
                   label="default (faded)"),
        plt.Line2D([0], [0], color="black", alpha=0.95, linewidth=8,
                   label="unified (solid)")
    ]

    axes[0].legend(
        handles=legend_handles,
        loc="upper left",
        fontsize=8,
        frameon=False
    )

    plt.subplots_adjust(wspace=0.02, left=0.05, right=0.995, top=0.83, bottom=0.20)
    plt.tight_layout(rect=[0.01, 0.05, 0.995, 0.85])

    # Single-line centered title: main (bold 14pt) + subtitle (plain 10pt) on
    # the same baseline. Render at x=0 first to measure widths, then center
    # the pair on the tight-bbox midpoint (so bbox_inches="tight" doesn't
    # shift the title off-center via asymmetric left/right cropping), and
    # nudge it slightly right of true centre to match the abundance plot.
    _short_title = DISPLAY_TITLE.replace("ZymoMockD6331", "Mock")
    _main = fig.text(0, 0.93,
                     f"Detection - {_short_title}",
                     ha="left", va="baseline", fontsize=14, fontweight="bold")
    _sub = fig.text(0, 0.93,
                    f" ({rank.capitalize()} level, threshold ≥ {THRESHOLD_PERCENT}%)",
                    ha="left", va="baseline", fontsize=10)
    fig.canvas.draw()
    _renderer = fig.canvas.get_renderer()
    _fig_w_in = fig.get_size_inches()[0]
    _fig_w_px = _fig_w_in * fig.dpi
    _main_w = _main.get_window_extent(renderer=_renderer).width / _fig_w_px
    _sub_w  = _sub.get_window_extent(renderer=_renderer).width  / _fig_w_px
    _tight = fig.get_tightbbox(_renderer)
    _center = ((_tight.x0 + _tight.x1) / 2) / _fig_w_in
    _shift_right = 0.025
    _left = _center - (_main_w + _sub_w) / 2 + _shift_right
    _main.set_x(_left)
    _sub.set_x(_left + _main_w)

    fig_path = FIGURES_DIR / f"{SAMPLE_LABEL}_{THRESHOLD_PERCENT}_metrics_{rank}.png"
    save_fig(fig, fig_path, SAVE, dpi=300, bbox_inches="tight")
    plt.show()

    if SAVE:
        print(f"Saved: {fig_path}")

print("\nAll visualizations complete!")

In [ ]:
# ---- Single-bar plot: unified bar + default tick + DB-shift arrow ----

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.transforms as mtransforms
import numpy as np

for rank in RANKS:
    print(f"\nGenerating single-bar plots for {rank}...")
    metrics_df = all_results[rank]["metrics"].copy()
    metrics_df[["base_tool", "db_type"]] = metrics_df["tool"].str.rsplit("_", n=1, expand=True)
    base_tools = [t for t in BASE_TOOL_ORDER if (metrics_df["base_tool"] == t).any()]

    fig, axes = plt.subplots(
        1, 3,
        figsize=(10, 4),
        sharey=True,
        gridspec_kw={"wspace": 0.04},
    )

    metric_names  = ["precision", "recall", "F1"]
    metric_titles = ["Precision", "Recall", "F1 Score"]
    x = np.arange(len(base_tools)) * 0.7

    for i, (ax, metric, title) in enumerate(zip(axes, metric_names, metric_titles)):
        ax.set_facecolor("#f2f2f2" if i % 2 == 0 else "white")

        for j, base in enumerate(base_tools):
            row_def = metrics_df[(metrics_df["base_tool"] == base) & (metrics_df["db_type"] == "default")]
            row_uni = metrics_df[(metrics_df["base_tool"] == base) & (metrics_df["db_type"] == "unified")]
            if row_def.empty or row_uni.empty:
                continue
            d_val = float(row_def[metric].iloc[0])
            u_val = float(row_uni[metric].iloc[0])
            delta = u_val - d_val
            color = TOOL_COLORS[base]

            ax.bar(x[j], u_val, width=0.5, color=color, alpha=0.9,
                   edgecolor="black", linewidth=0.4, zorder=2)
            ax.hlines(d_val, x[j] - 0.25, x[j] + 0.25,
                      colors="black", linestyles="--", linewidth=1.3, zorder=3)

            if abs(delta) >= 0.005:
                ax.annotate("",
                            xy=(x[j], u_val), xytext=(x[j], d_val),
                            arrowprops=dict(arrowstyle="-|>", color="black",
                                            lw=1.5, mutation_scale=11, alpha=0.9),
                            zorder=4)
                sign = "+" if delta > 0 else "−"
                y_sign = max(u_val - 0.05, 0.02)
                t = ax.text(x[j] + 0.18, y_sign, sign,
                            ha="center", va="center",
                            fontsize=10, fontweight="bold",
                            color="black", zorder=5)
                t.set_path_effects([pe.withStroke(linewidth=1.6, foreground="white")])

        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.set_xticks(x)
        ax.set_xticklabels(base_tools, rotation=65, ha="right", fontsize=10)
        _dx = mtransforms.ScaledTranslation(10 / 72, 0, fig.dpi_scale_trans)
        for _lbl in ax.get_xticklabels():
            _lbl.set_transform(_lbl.get_transform() + _dx)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.grid(axis="y", alpha=0.25)
        ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_ylim(0, 1.0)

    axes[0].set_ylabel("Score (unified DB)", fontsize=10)

    legend_handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor="gray", edgecolor="black",
                      linewidth=0.4, alpha=0.9, label="unified DB (bar)"),
        plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1.3,
                   label="default DB (tick)"),
        plt.Line2D([0], [0], color="black", marker=">", linestyle="-",
                   markersize=7, linewidth=1.5, label="DB shift"),
    ]
    axes[0].legend(handles=legend_handles, loc="upper left", frameon=False, fontsize=7.5)

    plt.subplots_adjust(wspace=0.02, left=0.05, right=0.995, top=0.83, bottom=0.20)
    plt.tight_layout(rect=[0.01, 0.05, 0.995, 0.85])

    _short_title = DISPLAY_TITLE.replace("ZymoMockD6331", "Mock")
    _main = fig.text(0, 0.93,
                     f"Detection - {_short_title}",
                     ha="left", va="baseline", fontsize=14, fontweight="bold")
    _sub = fig.text(0, 0.93,
                    f" ({rank.capitalize()} level, threshold ≥ {THRESHOLD_PERCENT}%)",
                    ha="left", va="baseline", fontsize=10)
    fig.canvas.draw()
    _renderer = fig.canvas.get_renderer()
    _fig_w_in = fig.get_size_inches()[0]
    _fig_w_px = _fig_w_in * fig.dpi
    _main_w = _main.get_window_extent(renderer=_renderer).width / _fig_w_px
    _sub_w  = _sub.get_window_extent(renderer=_renderer).width  / _fig_w_px
    _tight = fig.get_tightbbox(_renderer)
    _center = ((_tight.x0 + _tight.x1) / 2) / _fig_w_in
    _shift_right = 0.025
    _left = _center - (_main_w + _sub_w) / 2 + _shift_right
    _main.set_x(_left)
    _sub.set_x(_left + _main_w)

    fig_path = FIGURES_DIR / f"{SAMPLE_LABEL}_{THRESHOLD_PERCENT}_singlebar_delta_{rank}.png"
    save_fig(fig, fig_path, SAVE, dpi=300, bbox_inches="tight")
    plt.show()

    if SAVE:
        print(f"Saved: {fig_path}")


### Debug: missing ground-truth taxa
For each tool, show which GT taxa were not detected above `THRESHOLD_PERCENT`.


In [ ]:
# Which GT taxa did each tool miss at the detection threshold?
# (Mirrors the analysis_abundance.ipynb cell — answers the same question for detection.)

MISS_RANK = 'species'    # 'species' or 'genus'
TOOLS_TO_CHECK = TOOLS   # or a subset like ['Sourmash_unified', 'Sylph_unified']

payload = all_results[MISS_RANK]
gt_taxids = set(payload['ground_truth']['taxid'].astype(int))
thr = float(THRESHOLD_PERCENT)

for tool_db in TOOLS_TO_CHECK:
    df = payload['tool_tables'].get(tool_db)
    if df is None or df.empty:
        print(f'{tool_db}: no data'); continue
    pred_taxids = set(df.loc[df['value'] >= thr, 'taxid'].dropna().astype(int))
    missing = sorted(gt_taxids - pred_taxids)
    print(f'\n--- {tool_db} @ {MISS_RANK} ---')
    print(f'  recovered: {len(gt_taxids & pred_taxids)}/{len(gt_taxids)}  '
          f'missing: {len(missing)}')
    if missing:
        miss_df = payload['ground_truth'].query('taxid in @missing')[['taxid', 'name']]
        display(miss_df)
